In [9]:
!pip install -q segmentation-models-pytorch

In [10]:
import os
import gc
import time
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [11]:
# ==========================================
# 1. КОНФИГУРАЦИЯ И ПУТИ
# ==========================================
DATA_ROOT = Path(r"/kaggle/input/competitions/dl-lab-3-product-segmentation/train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR = DATA_ROOT / "masks"

# Пути к Псевдо-лейблам
PL_DIR = Path(r"/kaggle/input/datasets/tkachenko1van/pseudo-labels-confident/pseudo_labels_confident")
PL_IMAGES_DIR = PL_DIR / "images"
PL_MASKS_DIR = PL_DIR / "masks"

# Пути к твоим идеальным весам
OLD_WEIGHTS_CONF1 = Path(r"/kaggle/input/datasets/tkachenko1van/conf1-unetpp-effb4")
OLD_WEIGHTS_CONF2 = Path(r"/kaggle/input/datasets/tkachenko1van/conf2-manet-mitb3")
OLD_WEIGHTS_CONF3 = Path(r"/kaggle/input/datasets/tkachenko1van/conf3-deeplab-convnext")

# Папка для сохранения новых весов
SAVE_DIR = Path("/kaggle/working/finetuned_models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 320
BATCH_SIZE = 32
EPOCHS = 10
LR = 5e-5
ALPHA_NEW = 0.3

N_SPLITS = 5
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

In [12]:
# ==========================================
# 2. АУГМЕНТАЦИИ (Слегка смягчены для Fine-Tuning)
# ==========================================
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=15,
        border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.OneOf([
        A.ColorJitter(
            brightness=0.1,
            contrast=0.1,
            saturation=0.1,
            hue=0.02,
            p=1.0),
        A.RandomGamma(gamma_limit=(90, 110), p=1.0), 
    ], p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.GaussNoise(p=1.0),
    ], p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class SegDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = cv2.imread(str(self.df.loc[idx, 'image_path']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(str(self.df.loc[idx, 'mask_path']), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        return image, mask.unsqueeze(0)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [13]:
# ==========================================
# 3. ПОДГОТОВКА ДАТАФРЕЙМОВ (Orig + PL)
# ==========================================
def prepare_dataframes():
    print("📊 Сборка оригинального датафрейма...")
    orig_data = []
    for mask_path in sorted(MASKS_DIR.glob("*.png")):
        img_path = IMAGES_DIR / (mask_path.stem + ".jpg")
        if img_path.exists():
            mask_preview = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            orig_data.append({"image_path": str(img_path), "mask_path": str(mask_path), "area": np.sum(mask_preview > 0)})
    
    df_orig = pd.DataFrame(orig_data)
    df_orig['area_bin'] = pd.qcut(df_orig['area'], q=5, labels=False, duplicates='drop')
    
    print("🤖 Сборка датафрейма Псевдо-Лейблов...")
    pl_data = []
    for mask_path in sorted(PL_MASKS_DIR.glob("*.png")):
        img_path = PL_IMAGES_DIR / (mask_path.stem + ".jpg") # PL картинки обычно сохраняются как .jpg
        if img_path.exists():
            pl_data.append({"image_path": str(img_path), "mask_path": str(mask_path)})
            
    df_pl = pd.DataFrame(pl_data)
    print(f"Оригинал: {len(df_orig)} | Псевдо-Лейблы: {len(df_pl)}")
    return df_orig, df_pl

In [14]:
# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ И СМЕШИВАНИЯ (BLENDING)
# ==========================================
def dice_coef(y_pred, y_true, smooth=1e-6):
    y_pred = (torch.sigmoid(y_pred) > 0.5).float()
    intersection = (y_pred * y_true).sum(dim=(2, 3))
    union = y_pred.sum(dim=(2, 3)) + y_true.sum(dim=(2, 3))
    return ((2. * intersection + smooth) / (union + smooth)).mean().item()

def blend_weights(old_path, new_path, save_path, alpha=ALPHA_NEW):
    """Смешивает старые и новые веса (Защита от Catastrophic Forgetting)"""
    old_state = torch.load(old_path, map_location='cpu')
    new_state = torch.load(new_path, map_location='cpu')
    blended_state = {}
    
    for key in old_state.keys():
        blended_state[key] = (1.0 - alpha) * old_state[key] + alpha * new_state[key]
        
    torch.save(blended_state, save_path)
    print(f"🧬 Смешанные веса сохранены: {save_path.name}")

def train_one_fold(model, train_loader, val_loader, optimizer, scheduler, criterion, fold, conf_name, old_weights_path):
    scaler = torch.amp.GradScaler('cuda')
    best_dice = 0.0
    
    new_best_path = SAVE_DIR / f"{conf_name}_fold{fold}_finetuned.pth"
    blended_path  = SAVE_DIR / f"{conf_name}_fold{fold}_blended.pth"

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()
        model.train()
        train_loss = 0.0
        
        for images, masks in train_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        scheduler.step()

        model.eval()
        val_loss, val_dice = 0.0, 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = criterion(logits, masks)
                val_loss += loss.item()
                val_dice += dice_coef(logits, masks)
                
        val_loss /= len(val_loader)
        val_dice /= len(val_loader)

        epoch_time = time.time() - start_time
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), new_best_path)
            print(f"   Epoch {epoch:02d}/{EPOCHS} [{epoch_time:.0f}s] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f} 🌟")
        else:
            print(f"   Epoch {epoch:02d}/{EPOCHS} [{epoch_time:.0f}s] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")
            
    print(f"✅ Fold {fold} дообучен. Лучший Dice: {best_dice:.4f}.")
    
    # Делаем Weight Blending!
    blend_weights(old_weights_path, new_best_path, blended_path)


def run_finetune(df_orig, df_pl, conf_name, arch, encoder, old_weights_dir):
    print(f"\n{'='*50}\n🚀 FINE-TUNING: {conf_name}\n{'='*50}")
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    
    # НОВЫЙ МОЩНЫЙ LOSS: 0.4 Dice + 0.4 Focal + 0.2 Boundary
    dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)
    focal_loss = smp.losses.FocalLoss(mode='binary')
    combo_loss = lambda p, t: 0.5 * dice_loss(p, t) + 0.5 * focal_loss(p, t)

    for fold, (train_idx, val_idx) in enumerate(skf.split(df_orig, df_orig['area_bin']), 1):
        old_weights_path = old_weights_dir / f"{conf_name}_fold{fold}.pth"
        if not old_weights_path.exists():
            print(f"⚠️ Пропуск Фолд {fold}: Старые веса не найдены!")
            continue

        print(f"\n--- Обучение Fold {fold}/{N_SPLITS} ---")
        
        # ❗ ВАЖНО: Склеиваем оригинальный трейн с PL. Валидация остается чистой!
        train_orig = df_orig.iloc[train_idx]
        val_df = df_orig.iloc[val_idx]
        train_df = pd.concat([train_orig, df_pl]).reset_index(drop=True)
        
        train_loader = DataLoader(SegDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(SegDataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        
        # Инициализация архитектуры
        if arch == "UnetPlusPlus":
            model = smp.UnetPlusPlus(encoder_name=encoder, encoder_weights=None, in_channels=3, classes=1)
        elif arch == "MAnet":
            model = smp.MAnet(encoder_name=encoder, encoder_weights=None, in_channels=3, classes=1)
        elif arch == "DeepLabV3Plus":
            model = smp.DeepLabV3Plus(encoder_name=encoder, encoder_weights=None, in_channels=3, classes=1)
            
        # Загружаем старые идеальные веса перед дообучением
        model.load_state_dict(torch.load(old_weights_path, map_location='cpu'))
        model.to(DEVICE)
        
        # Оптимизатор и плавно затухающий микро-LR
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
        
        train_one_fold(model, train_loader, val_loader, optimizer, scheduler, combo_loss, fold, conf_name, old_weights_path)
        
        del model, optimizer, scheduler, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

In [15]:
# ==========================================
# 6. ЗАПУСК ДООБУЧЕНИЯ ВСЕХ КОНФИГУРАЦИЙ
# ==========================================
df_orig, df_pl = prepare_dataframes()

# 1. Unet++
run_finetune(df_orig, df_pl, "conf1_unetpp_effb4", "UnetPlusPlus", "efficientnet-b4", OLD_WEIGHTS_CONF1)

# 2. MAnet
run_finetune(df_orig, df_pl, "conf2_manet_mitb3", "MAnet", "mit_b3", OLD_WEIGHTS_CONF2)

# 3. DeepLabV3+
run_finetune(df_orig, df_pl, "conf3_deeplab_convnext", "DeepLabV3Plus", "tu-convnext_small", OLD_WEIGHTS_CONF3)

print("\n🎉 FINE-TUNING ПОЛНОСТЬЮ ЗАВЕРШЕН! Смешанные веса лежат в /kaggle/working/finetuned_models/")

📊 Сборка оригинального датафрейма...
🤖 Сборка датафрейма Псевдо-Лейблов...
Оригинал: 2000 | Псевдо-Лейблы: 284

🚀 FINE-TUNING: conf1_unetpp_effb4

--- Обучение Fold 1/5 ---
   Epoch 01/10 [74s] | Train Loss: 0.0274 | Val Loss: 0.0605 | Val Dice: 0.9107 🌟
   Epoch 02/10 [74s] | Train Loss: 0.0268 | Val Loss: 0.0608 | Val Dice: 0.9110 🌟
   Epoch 03/10 [74s] | Train Loss: 0.0262 | Val Loss: 0.0603 | Val Dice: 0.9112 🌟
   Epoch 04/10 [75s] | Train Loss: 0.0265 | Val Loss: 0.0605 | Val Dice: 0.9113 🌟
   Epoch 05/10 [75s] | Train Loss: 0.0263 | Val Loss: 0.0603 | Val Dice: 0.9109
   Epoch 06/10 [75s] | Train Loss: 0.0258 | Val Loss: 0.0605 | Val Dice: 0.9107
   Epoch 07/10 [75s] | Train Loss: 0.0257 | Val Loss: 0.0608 | Val Dice: 0.9107
   Epoch 08/10 [75s] | Train Loss: 0.0255 | Val Loss: 0.0610 | Val Dice: 0.9104
   Epoch 09/10 [75s] | Train Loss: 0.0256 | Val Loss: 0.0608 | Val Dice: 0.9107
   Epoch 10/10 [74s] | Train Loss: 0.0255 | Val Loss: 0.0609 | Val Dice: 0.9105
✅ Fold 1 дообучен. 